In [ ]:
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import AsyncOpenAI, OpenAI
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import matplotlib.pyplot as plt
from difflib import SequenceMatcher
from typing import Optional
import json
import yaml
from typing import Optional, Any
import os
import asyncio

# Introdución

Llegados a este punto tenemos el texto de los documentos del corpus ya extraído y procesado, y las preguntas de single-turn extraídas de cada documento y agrupadas en un mismo archivo en formato jsonL.

En el presente notebook vamos a generar preguntas multiturno a partir de algunas de las preguntas single-turn. Concretamente tenemos 134 preguntas single-turn entre las 4 categorías, factual_true, factual_reasoning, factual_trap y out_of_domain.

El objetivo para este notebook es generar alrededor de 20 preguntas multiturno de 3 turnos.La proporción estándar en la literatura es de 15%-20% del total del dataset de preguntas. Aunque por ejemplo en el paper de MT-EVal, se usan 168 diálogosmultiturno con una media de unos 7 turnos por diálogo, con 3 turnos por conversación es suficientem para capturar degradación por distancia y error de propagación en las respuestas del RAG (son dos factores clave indentificados por MT-EVAl)

MT-EVAl indentifica 4 patrones de interacción real que serán en los que nso vamos a apoyar para la generación de las preguntas multiturno:
| Tipo MT-Eval | Descripción                                       | Categorías de tu dataset compatibles      |
| ------------ | ------------------------------------------------- | ----------------------------------------- |
| Follow-up (seguir)   | Cada turno se basa en la respuesta anterior       | factual_true, factual_reasoning           |
| Expansion  (ampliar)   | Distintas preguntas sobre el mismo tema/documento | factual_true + factual_trap del mismo doc |
| Refinement (precisar)  | El usuario añade restricciones progresivas        | factual_reasoning                         |
| Recollection (memoria) | El modelo debe recordar info de turnos anteriores | factual_trap                              |





**¿Qué es una pregunta multiturno?
¿Porqué es conveniente que evaluemos también el RAG y los LLMs baseline con preguntas multiturno? ¿Qué nos permite ver una pregunta multiturno que no nos permitan las preguntas single-turn?**

Las preguntas multiturno nos permiten simular una conversación real con el RAG.


**¿Debo descartar aquellas preguntas single-turn a partir de las cuales haya construido las preguntas multi-turn?**


Esta es una dudad metodológica importante, y su respuesta es claramente NO.
Las preguntas multiturno son ítems nuevos e independientes en el dataset. La pregunta single-turn original sigue siendo válida como ítem de evaluación por sí misma. De hecho, esto es exactamente lo que hace MT-Eval: construye versiones single-turn de cada turno de sus conversaciones multiturno precisamente para medir la degradación de rendimiento entre ambos contextos.

Para el TFG puede ser incluso una cierta ventaja analítica:

| Evaluación             | Qué mides                                                      |
| ---------------------- | -------------------------------------------------------------- |
| Single-turn seed       | Capacidad del LLM/RAG sobre pregunta aislada                   |
| Multiturno derivado    | Misma pregunta como turno N de conversación                    |
| Diferencia entre ambos | Degradación por contexto conversacional → resultado publicable |

Tener la misma pregunta en ambos formatos te permite medir exactamente el coste de la historia conversacional sobre la fidelidad, que es una contribución diferencial de tu TFG respecto a trabajos que solo evalúan single-turn.

# Primer Tipo de Preguntas Multiturno
---
Single-turn --> Multi-turn


 La idea es que el primer turno de la conversación sea la propia pregunta semilla extraída del golden dataset de preguntas single-turn, y que a partir de dicha pregunta, y su contexto completo (todo el documento) el LLM sea capaz de generar una conversación que simule un diálogo que un empleado de AMC pudiera tener con el sistema RAG.

##1. Escoger las preguntas semilla single-turn a partir de las cuales vamos a generar las preguntas multiturno.

Como primer paso debemos recordar que las preguntas out_of_domain, no nos servirán para construir a partir de ellas las multiturno, ya que estaríamos generando preguntas que no estarían basadas en información factual del corpus.

Vamos a seguir un criterio de scoring para la selección de la preguntas semilla. Esta es una decisión metodológica que es justificable y replicable para la memoria. Concretamente diseño una función de scoring automático con ciertos criterios explícitos.

Los criterios de scoring que seguirá nuestra función son los siguientes:

| Criterio                                     | Peso | Justificación                                                                           |
| -------------------------------------------- | ---- | --------------------------------------------------------------------------------------- |
| Riqueza del context_gold (longitud)          | 30%  | Contextos más largos ofrecen más entidades y relaciones para generar turnos adicionales |
| Categoría                                    | 25%  | factual_reasoning > factual_true                               |
| Densidad de entidades nombradas              | 25%  | Más entidades = más ángulos para Follow-up y Expansion                                  |
| Diversidad respecto a seeds ya seleccionados | 20%  | Evita seeds redundantes; maximiza cobertura temática del documento                      |

Para este primer tipo de preguntas multiturno vamos a excluir tanto a las preguntas out_of_domain ya que no tienen context_gold a partir del cual podamos fundamentar factuamente sus respuestas, como a las preguntas factual_trap, ya que darían lugar a iniciar la conversación con una premisa falsa, y el LLM generador podría construir los turnos de la conversación siguiendo esa base errónea.

("Se excluyeron como semillas las preguntas de categoría factual_trap y out_of_domain, dado que las primeras contienen premisas intencionalmente falsas que invalidarían la coherencia de los turnos generados, y las segundas carecen de contexto documental de referencia.")

Luego respecto a las otras dos categorías de preguntas single-turn, cabe destacar que factual_reasoning sigue siendo preferible a factual_true como seed porque su context_gold es más rico en relaciones causales, lo que da más material al LLM para generar turnos de Follow-up y Refinement con profundidad.



```
CATEGORY_SCORE = {
    "factual_reasoning": 1.0,
    "factual_true":      0.7,
    "factual_trap":      0.0,  # Excluida: premisa falsa no es válida como seed
    "out_of_domain":     0.0,  # Excluida: sin context_gold real
}

```



En el siguiente código se aborda la selección de las preguntas semilla (seeds) mediante una función de scoring que asigna una puntuación de 0 a 100 a cada candidata. Los cuatro criterios que componen dicha puntuación se calculan de la siguiente forma:

**-Riqueza del contexto:** Se mide como la longitud en caracteres del context_gold completo de la pregunta, concatenando todos sus fragmentos en caso de que existan varios separados por [...]. Este valor se normaliza al rango estableciendo un techo de 300 caracteres, bajo la suposición de que un fragmento de esa extensión ya contiene información suficiente para sostener una conversación de varios turnos. Fragmentos más cortos penalizan la puntuación proporcionalmente, ya que ofrecen menos material al LLM generador para construir turnos adicionales coherentes.


**-Categoría de la pregunta:** Solo se consideran elegibles como seed las categorías factual_true y factual_reasoning, que son aquellas cuya premisa es verdadera y está respaldada por el documento. Las preguntas de categoría factual_trap quedan excluidas porque contienen una premisa intencionalmente falsa: usarlas como punto de partida haría que el LLM construyese los turnos siguientes sobre una base errónea, comprometiendo la coherencia y validez de toda la conversación generada. Las preguntas out_of_domain se excluyen igualmente al carecer de context_gold documental de referencia. Dentro de las categorías elegibles, factual_reasoning recibe una puntuación máxima (1.0) frente a factual_true (0.85), dado que sus contextos tienden a contener relaciones causales y comparativas que ofrecen mayor riqueza para la generación de turnos de tipo Follow-up y Refinement.


**-Reconocimiento de entidades nombradas:** Una entidad nombrada es cualquier mención en el texto que se refiere a algo concreto y específico del mundo real: organizaciones (AMC, CITRUSPACK), cantidades (60.000 toneladas, 20%), fechas (diciembre de 2019), lugares (España, Francia) o productos (CitrusPLA). Para su detección se realiza una llamada al modelo gpt-4o-mini de OpenAI, al que se le solicita exclusivamente un recuento de entidades nombradas sobre el context_gold completo. Esta aproximación resulta más robusta que modelos NER estáticos como los de spaCy para textos técnicos de dominio industrial, ya que el modelo comprende el contexto semántico del fragmento y distingue correctamente entidades como magnitudes de producción o nombres de proyectos propios del dominio de AMC. El recuento obtenido se normaliza dividiendo entre 5, fijando así que un fragmento con 5 o más entidades nombradas se considera suficientemente rico a efectos de este criterio. Cuantas más entidades contiene un fragmento, más ángulos distintos ofrece al LLM generador para construir preguntas de seguimiento del tipo "¿Y qué ocurrió con AMC en Francia?" o "¿Cuándo exactamente se realizaron esas pruebas?".


**-Diverisad respecto a preguntas ya evaluadas:** Se busca que el conjunto final de preguntas seedsm (preguntas indiviuales a partir de las cuales se construye el diálogo multi-turno) sea temáticamente diverso y no redundante. Para ello, se obtiene el embedding semántico del context_gold de la pregunta candidata usando el modelo text-embedding-3-small de OpenAI, y se compara mediante similitud coseno contra los embeddings de los contextos de todas las seeds ya seleccionadas. La diversidad se define como el complemento de la similitud máxima observada: una candidata muy similar a alguna seed ya elegida recibe una puntuación baja en este criterio, desincentivando la selección de preguntas redundantes.


La selección sigue un algoritmo greedy secuencial: en cada iteración se elige la candidata con mayor puntuación global considerando las seeds ya confirmadas, lo que garantiza que la diversidad se evalúa de forma acumulativa y no en aislamiento.



In [ ]:
def get_embedding(text: str, client: OpenAI, model: str = "text-embedding-3-small") -> list[float]:
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In [ ]:
def count_entities_openai(context: str, client: OpenAI) -> int:
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Barato y suficiente para esta tarea
        messages=[{
            "role": "user",
            "content": (
                "Count the number of named entities in the following text "
                "(organizations, dates, quantities, locations, products). "
                "Respond ONLY with an integer.\n\n"
                f"Text: {context}"
            )
        }],
        max_tokens=5,
        temperature=0
    )
    try:
        return int(response.choices[0].message.content.strip())
    except ValueError:
        return 0


In [ ]:
def precompute_static_scores(candidates: list[dict], client: OpenAI) -> dict:
    """Precomputa richness, cat_score y embedding una sola vez por candidata."""
    cache = {}
    for q in candidates:
        qid = q["metadata"]["id"]
        context = " [...] ".join(q["reference_contexts"])
        cache[qid] = {
            "question": q,
            "richness":  min(len(context) / 300, 1.0),
            "cat_score": CATEGORY_SCORE.get(q["metadata"]["category"], 0.0),
            "n_entities": count_entities_openai(context, client),  # 1 sola vez
            "embedding":  np.array(get_embedding(context, client)),  # 1 sola vez
            "context":    context,
        }
    return cache

In [ ]:


def select_seeds(all_questions: list[dict], client: OpenAI, n_seeds: int = 20) -> list[dict]:
    candidates = [q for q in all_questions
                  if CATEGORY_SCORE.get(q["metadata"]["category"], 0.0) > 0]

    cache = precompute_static_scores(candidates, client)

    selected = []
    selected_embs = []

    for _ in range(n_seeds):
        best_q, best_score = None, -1

        for q in candidates:
            if q in selected:
                continue
            qid = q["metadata"]["id"]
            c = cache[qid]

            if c["cat_score"] == 0.0:
                continue

            entity_density = min(c["n_entities"] / 5, 1.0)

            if not selected_embs:
                diversity = 1.0
            else:
                emb = c["embedding"].reshape(1, -1)
                sims = cosine_similarity(emb, np.array(selected_embs))
                diversity = 1.0 - sims.max()

            score = (0.30 * c["richness"] +
                     0.25 * c["cat_score"] +
                     0.25 * entity_density +
                     0.20 * diversity)

            if score > best_score:
                best_score, best_q = score, q

        if best_q is None:
            break

        best_q["score"] = round(best_score * 100, 2)  # fix KeyError en plot
        selected.append(best_q)
        selected_embs.append(cache[best_q["metadata"]["id"]]["embedding"])

    return selected

In [ ]:
def plot_seeds_ranking(selected_seeds: list[dict]) -> None:
    ids    = [q["metadata"]["id"] for q in selected_seeds]
    scores = [q["score"] for q in selected_seeds]

    # Ordenar de mayor a menor
    pairs  = sorted(zip(scores, ids), reverse=True)[:10]
    scores_sorted = [p[0] for p in pairs]
    ids_sorted    = [p[1] for p in pairs]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(ids_sorted[::-1], scores_sorted[::-1], color="#6366f1")

    # Etiqueta de score al final de cada barra
    for bar, score in zip(bars, scores_sorted[::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{score:.1f}", va="center", fontsize=10)

    ax.set_xlabel("Score (0–100)")
    ax.set_ylabel("ID Pregunta")
    ax.set_title("Top-10 Mejores prehuntas para actuar como Seeds")
    ax.set_xlim(0, 110)
    plt.tight_layout()
    plt.savefig("seeds_ranking.png", dpi=150)
    plt.show()


# 2. Prompt





Generamos un prompt adecuado para esta tarea, el cual debe contener la pregunta single-turn, su contexto y el texto original del documento para que el LLM tenga contexto a partir del cual completar la pregunta.


Patrones conversacionales multiturno
---
La generación de conversaciones multiturno en este trabajo sigue la taxonomía de patrones conversacionales propuesta por Kwan et al. (2024) en MT-Eval, un benchmark diseñado específicamente para evaluar las capacidades de los LLMs en interacciones de múltiples turnos. A partir del análisis de conversaciones reales entre usuarios y asistentes en el corpus LMSYS-Chat-1M, los autores identificaron y formalizaron cuatro patrones predominantes que caracterizan la mayoría de interacciones reales con asistentes de IA:
​

Follow-up (seguimiento): el usuario formula preguntas que se construyen directamente sobre la respuesta inmediatamente anterior del asistente, referenciando elementos específicos de dicha respuesta. Estas preguntas son semánticamente dependientes del turno previo y no pueden interpretarse correctamente sin leerlo. Este patrón evalúa la capacidad del modelo de mantener coherencia referencial en conversaciones encadenadas.

Expansion (expansión): el usuario explora distintos aspectos o subtemas dentro del mismo dominio temático introducido en el primer turno, sin referenciar explícitamente las respuestas anteriores. Cada pregunta es temáticamente autónoma pero permanece dentro del mismo marco documental. Este patrón evalúa la amplitud de cobertura que un modelo puede ofrecer sobre un documento dado.

Refinement (refinamiento): el usuario clarifica o restringe progresivamente su petición original, añadiendo nuevas condiciones, formatos o ámbitos en cada turno. Las restricciones se acumulan: cada respuesta correcta debe satisfacer simultáneamente todas las constraints introducidas hasta ese punto. Este patrón evalúa la capacidad del modelo de mantener y actualizar instrucciones compuestas a lo largo de la conversación.

Recollection (recuerdo): el usuario establece una instrucción global o una condición de referencia en el primer turno y, después de varios turnos intermedios sobre temas relacionados, formula una pregunta que solo puede responderse correctamente si el modelo ha retenido y aplicado la instrucción original. La distancia entre la instrucción inicial y el turno que la pone a prueba es el factor determinante de la dificultad. Este patrón evalúa la memoria conversacional a largo plazo y la resistencia a la propagación de errores. Debe tener un número de turnos mayor que en los demás casos. Además no tendrá pregunta seed de la que se extraerá.
​

MT-Eval reporta que la mayoría de los LLMs evaluados presentan una degradación significativa de rendimiento en entornos multiturno respecto a sus equivalentes single-turn, siendo el patrón Recollection el que produce las caídas más pronunciadas en casi todos los modelos, seguido de Refinement. Estos resultados fundamentan la pertinencia de incluir precisamente estos cuatro patrones en el dataset de evaluación del presente trabajo.


Diseño de los prompts de generación y metadatos de trazabilidad
---
Para la generación automática de conversaciones multiturno se ha diseñado un prompt específico por cada uno de los cuatro patrones descritos, siguiendo el mismo enfoque que MT-Eval, que utiliza GPT-4 con instrucciones diferenciadas para cada categoría conversacional con el fin de garantizar que las conversaciones generadas sean representativas del patrón objetivo y no mezclen comportamientos de distintos tipos. Los prompts de Follow-up, Expansion y Refinement comparten una base común —identidad corporativa de AMC Global, sección documental de referencia y turno semilla fijo— pero difieren en la sección de instrucciones específicas, que define con precisión qué debe y qué no debe hacer el LLM generador en cada turno adicional.
​
El prompt de Recollection constituye una excepción justificada a este diseño común. El patrón Recollection requiere por definición que el primer turno establezca una instrucción global persistente que el asistente deba retener y aplicar a lo largo de toda la conversación, algo estructuralmente incompatible con una pregunta factual ordinaria, que es la naturaleza de todas las seeds del dataset. Forzar una seed factual como Turn 1 de una conversación Recollection produciría conversaciones artificiales que no respetan el patrón tal como lo define Kwan et al. (2024). Por este motivo, el prompt de Recollection no utiliza seed: tanto el Turn 1 como los turnos sucesivos se generan íntegramente a partir de la sección documental de referencia, permitiendo al LLM generador construir una instrucción global coherente y natural en el primer turno antes de proceder con los turnos de distancia y prueba.

Adicionalmente, los prompts de Refinement y Recollection incorporan campos de metadatos estructurados en la salida JSON que van más allá de la mera generación de texto. En el caso de Refinement, cada turno incluye el campo constraint_added, que registra de forma explícita y categorizada la nueva restricción introducida en ese turno. Esto permite, durante la fase de evaluación, detectar automáticamente qué tipo de constraint (de ámbito, de formato, de actor, de periodo temporal, etc.) resulta más susceptible de ser ignorada por cada LLM evaluado, sin necesidad de revisión manual turno a turno.

En el caso de Recollection, los turnos se etiquetan con dos roles diferenciados: bridge, para los turnos intermedios cuya función es crear distancia conversacional entre la instrucción global y el turno de prueba, y recollection-test, para el turno que realmente evalúa si el modelo ha retenido la información del primer turno. Esta distinción permite filtrar los turnos críticos de evaluación durante el cálculo de métricas con RAGAS, evitando que los turnos puente —que no someten a prueba la memoria del modelo— diluyan los resultados y oculten el fenómeno que se quiere medir: la degradación de fidelidad en función de la distancia al contexto relevante, identificado por Kwan et al. (2024) como uno de los factores principales de degradación en entornos multiturno.


Referencia APA 7 para incluir al final de la memoria:

Kwan, W., Zeng, X., Jiang, Y., Wang, Y., Li, L., Shang, L., Jiang, X., Liu, Q., y Wong, K. (2024). MT-Eval: A multi-turn capabilities evaluation benchmark for large language models. arXiv:2401.16745.

Para la generación de conversaciones multiturno, el LLM generador necesita disponer de suficiente material documental como para construir varios turnos coherentes y temáticamente consistentes. Usar exclusivamente el reference_context de la pregunta semilla —un fragmento de 2 a 4 frases anotado por expertos— resulta insuficiente: el modelo generador agota el material disponible en el segundo turno y comienza a repetir información o, en el peor caso, a alucinar contenido no presente en el documento. Sin embargo, proporcionar el documento completo introduce el riesgo opuesto: el modelo puede construir turnos basados en secciones semánticamente alejadas del fragmento original, lo que haría que el context_gold de los turnos generados resultara imposible de determinar de forma automática.

Para resolver este compromiso se propone una estrategia de ampliación por ventana de caracteres sobre el reference_context. La idea consiste en localizar cada fragmento anotado dentro del texto completo del documento y extraer una ventana simétrica de ±w caracteres alrededor de él, obteniendo así una sección documental acotada que incluye el fragmento de referencia y su contexto inmediato. Este enfoque presenta dos ventajas metodológicas relevantes: en primer lugar, no requiere un preprocesamiento previo del documento en secciones o chunks, lo que elimina una fuente adicional de decisiones de diseño arbitrarias; en segundo lugar, garantiza que el material proporcionado al LLM generador esté semánticamente centrado en el fragmento que originó la pregunta semilla, manteniendo la coherencia temática a lo largo de todos los turnos generados.

En los casos en que la pregunta semilla dispone de múltiples fragmentos en reference_contexts —es decir, cuando la anotación humana identificó evidencia distribuida en distintas partes del documento—, la función aplica la ampliación de forma independiente sobre cada fragmento y concatena los resultados mediante el separador [...]. Adicionalmente, se aplica un mecanismo de deduplicación basado en similitud de secuencias que descarta fragmentos ampliados cuyo inicio solapa significativamente con el final del fragmento anterior, evitando que el LLM generador reciba el mismo pasaje textual duplicado con el consiguiente sesgo en la generación. El umbral de solapamiento y el tamaño de ventana son hiperparámetros configurables, fijados en w = 1.500 caracteres y un ratio de similitud de 0.5 como valores de partida tras una inspección exploratoria del corpus documental de AMC Global.

In [ ]:
def get_generation_context(question: dict, full_doc_text: str,
                           window: int = 1500) -> str:
    """
    Para cada fragmento en reference_contexts, localiza su posición en el documento
    y lo amplía con una ventana de ±window caracteres de forma independiente.
    Los fragmentos ampliados se concatenan con [...] como separador.
    Los fragmentos con solapamiento significativo se deduplicán.
    """
    from difflib import SequenceMatcher

    def _significant_overlap(a: str, b: str, threshold: float = 0.5) -> bool:
        return SequenceMatcher(None, a[-200:], b[:200]).ratio() > threshold

    expanded_fragments = []

    for fragment in question["reference_contexts"]:

        search_key = fragment[:80]
        start_pos  = full_doc_text.find(search_key)

        if start_pos == -1:
            expanded_fragments.append(fragment)
            continue

        context_start = max(0, start_pos - window)
        context_end   = min(len(full_doc_text), start_pos + len(fragment) + window)

        expanded = full_doc_text[context_start:context_end]

        if context_start > 0:
            first_space = expanded.find(" ")
            expanded = "..." + expanded[first_space + 1:]

        if context_end < len(full_doc_text):
            last_space = expanded.rfind(" ")
            expanded = expanded[:last_space] + "..."

        expanded_fragments.append(expanded)

    # Deduplicación: descartar fragmento si su inicio solapa con el final del anterior
    deduplicated = [expanded_fragments[0]] if expanded_fragments else []
    for frag in expanded_fragments[1:]:
        if not _significant_overlap(deduplicated[-1], frag):
            deduplicated.append(frag)

    return " [...] ".join(deduplicated)



## Peticiones a la API de OpenAI.
---

La idea es poder hacer las llamadas de forma asíncrona a la API de OpenAI a la hora de generar las preguntas multiturno. Pretendo que a partir de una misma pregunta semilla se generen 4 preguntas multiturno, cada una conforme a un patrón distinto.

 El patrón Recollection se ha configurado con un número de turnos superior al resto de patrones (5 frente a 3) siguiendo la evidencia empírica reportada por Kwan et al. (2024). Los autores demuestran que el rendimiento de los modelos en este patrón decrece de forma consistente a medida que aumenta la distancia entre el turno de instrucción global y el turno que la pone a prueba, siendo este factor —la distancia al contenido relevante— uno de los dos determinantes principales de la degradación multiturno identificados en MT-Eval. Con un total de 5 turnos (1 de instrucción, 3 de puente y 1 de prueba de recuerdo), se introducen 3 turnos intermedios que crean una distancia conversacional suficiente para que el fenómeno de olvido de instrucción sea observable, superando el umbral mínimo de 1 turno de distancia con el que la degradación no resulta estadísticamente significativa. MT-Eval utiliza hasta 10 turnos por diálogo en este patrón; la configuración de 5 turnos adoptada en el presente trabajo constituye un compromiso entre la representatividad del fenómeno y las restricciones de coste de generación propias de un corpus de dominio cerrado.


In [ ]:
def add_extra_fields(
    turns: list[dict],
    tipo_pregunta: str,
    pregunta_semilla: Optional[dict],
    fragmento_generacion: str
) -> list[dict]:
    """
    Añade campos de trazabilidad a cada turno generado:
    patrón conversacional, id y documento de origen de la seed,
    y longitud del fragmento de generación usado.
    """
    if pregunta_semilla is not None:
        seed_id  = pregunta_semilla["metadata"]["id"]
        seed_doc = pregunta_semilla["metadata"].get("doc_name", "unknown")
    else:
        # Recollection no tiene seed
        seed_id  = None
        seed_doc = None

    for turn in turns:
        turn.setdefault("metadata", {})
        turn["metadata"]["pattern"]                   = tipo_pregunta
        turn["metadata"]["seed_id"]                   = seed_id
        turn["metadata"]["seed_doc"]                  = seed_doc
        turn["metadata"]["generation_context_length"] = len(fragmento_generacion)

    return turns


In [ ]:
async def obtener_preguntas(
    client: AsyncOpenAI,
    prompt: str,
    tipo_pregunta: str,
    fragmento_generacion: str,
    n_turns: int,
    temp: float,
    pregunta_semilla: Optional[dict] = None,
    model: str = "gpt-4.1"
) -> list[dict]:
    """
    Llamada asíncrona a la API de OpenAI para generar una conversación multiturno.
    Recollection no recibe pregunta_semilla (None por defecto).
    """

    # Recollection genera Turn 1 desde cero → no necesita seed_question ni seed_answer
    format_kwargs = {
        "generation_context": fragmento_generacion,
        "n_turns": n_turns,
    }
    if tipo_pregunta != "recollection":
        format_kwargs["seed_question"] = pregunta_semilla["user_input"]
        format_kwargs["seed_answer"]   = pregunta_semilla["reference"]

    response = await client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": prompt.format(**format_kwargs)
        }],
        temperature=temp,
        response_format={"type": "json_object"}
    )

    generated_turns = json.loads(response.choices[0].message.content)["turns"]

    # Recollection genera Turn 1 por sí mismo → no añadimos nada
    if tipo_pregunta == "recollection":
        all_turns = generated_turns

    # Resto de patrones → Turn 1 es la seed, se antepone a los turnos generados
    else:
        turn_1 = {
            "turn": 1,
            "user": pregunta_semilla["user_input"],
            "assistant": pregunta_semilla["reference"],
            "source_fragment": " [...] ".join(pregunta_semilla["reference_contexts"])
        }
        all_turns = [turn_1] + generated_turns

    return add_extra_fields(all_turns, tipo_pregunta, pregunta_semilla, fragmento_generacion)



In [ ]:
def load_document(doc_name: str, doc_index: dict[str, str]) -> str:
    """
    Devuelve el texto_completo del documento a partir del índice preconstruido.
    """
    if doc_name not in doc_index:
        raise FileNotFoundError(
            f"'{doc_name}' no encontrado en el índice. "
            f"Claves disponibles: {list(doc_index.keys())}"
        )
    with open(doc_index[doc_name], "r", encoding="utf-8") as f:
        return json.load(f)["texto_completo"]

In [ ]:
async def generate_all(seeds, prompts, client_async, doc_index):
    tasks = []
    meta  = []
    gen_params = prompts["generation_params"]

    for seed in seeds:
        doc_text   = load_document(seed["metadata"]["doc_name"], doc_index)
        fragmento  = get_generation_context(seed, doc_text)

        for pattern in ["follow-up", "expansion", "refinement", "recollection"]:
            tasks.append(obtener_preguntas(
                client     = client_async,
                prompt     = prompts[f"prompt_multiturno_{pattern.replace('-', '')}"],
                tipo_pregunta    = pattern,
                fragmento_generacion = fragmento,
                n_turns    = 3 if pattern != "recollection" else 5,
                temp       = gen_params[pattern]["temperature"],
                pregunta_semilla = seed if pattern != "recollection" else None
            ))
            meta.append({"seed_id": seed["metadata"]["id"], "pattern": pattern})

    results = await asyncio.gather(*tasks)

    conversations = [
        {
            "conversation_id": f"{m['seed_id']}_{m['pattern']}",
            "pattern":         m["pattern"],
            "seed_id":         m["seed_id"],
            "turns":           turns
        }
        for m, turns in zip(meta, results)]

    return conversations


# Main
---

In [ ]:
def export_conversations_readable(
    jsonl_path: str,
    output_dir: str
) -> None:
    """
    Lee el JSONL de conversaciones multiturno y guarda cada una
    en un fichero .txt individual legible por un humano.
    """
    export_dir = os.path.join(output_dir, "multiturno-individuales")
    os.makedirs(export_dir, exist_ok=True)

    with open(jsonl_path, "r", encoding="utf-8") as f:
        conversations = [json.loads(line) for line in f if line.strip()]

    for conv in conversations:
        conv_id = conv.get("conversation_id", "unknown")
        pattern = conv.get("pattern", "unknown")
        seed_id = conv.get("seed_id", "N/A")
        turns   = conv.get("turns", [])

        lines = []
        lines.append("=" * 60)
        lines.append(f"  CONVERSACIÓN: {conv_id}")
        lines.append(f"  Patrón:       {pattern.upper()}")
        lines.append(f"  Seed ID:      {seed_id}")
        lines.append("=" * 60)

        for turn in turns:
            turn_num = turn.get("turn", "?")
            turn_type = turn.get("type", "")          # solo Recollection tiene 'type'
            user_msg  = turn.get("user", "")
            assistant_msg = turn.get("assistant", "")
            source    = turn.get("source_fragment", "")
            constraint = turn.get("constraint_added", "")  # solo Refinement

            # Cabecera del turno
            header = f"\n[TURNO {turn_num}]"
            if turn_type:
                header += f"  ({turn_type})"
            if constraint:
                header += f"  → constraint: {constraint}"
            lines.append(header)
            lines.append("-" * 40)

            lines.append(f" EMPLEADO:\n   {user_msg}")
            lines.append(f"\n ASISTENTE:\n   {assistant_msg}")
            lines.append(f"\n FUENTE:\n   {source}")

        lines.append("\n" + "=" * 60 + "\n")

        # Nombre del fichero: {conversation_id}.txt
        filename  = f"{conv_id}.txt"
        file_path = os.path.join(export_dir, filename)
        with open(file_path, "w", encoding="utf-8") as out:
            out.write("\n".join(lines))

    print(f" {len(conversations)} conversaciones exportadas en: {export_dir}")


In [ ]:
def build_doc_index(docs_path: str) -> dict[str, str]:
    """
    Construye un diccionario {doc_name → ruta_completa} una sola vez.
    El fichero sigue el patrón D-{n}-{doc_name}_questions.json,
    por lo que se extrae el doc_name quitando el prefijo 'D-{n}-' y el sufijo '_questions.json'.
    """
    index = {}
    for filename in os.listdir(docs_path):
        if not filename.endswith(".pdf.json"):
            continue
        sin_sufijo  = filename.replace(".json", "")
        doc_name    = "-".join(sin_sufijo.split("-")[2:])
        index[doc_name] = os.path.join(docs_path, filename)
    return index

In [ ]:
#Función para la extracción de la configuración desde el archivo .yml
def load_config(config_file): # Parameterize config file name
    """
    Carga la configuración desde el fichero YAML.
    """
    try:
        with open(config_file, "r") as f:
            config = yaml.safe_load(f)
        print(" Configuración cargada exitosamente.")
        return config
    except Exception as e:
        print(f" Error al cargar la configuración: {e}")
        raise

In [ ]:
CATEGORY_SCORE = {
    "factual_reasoning": 1.0,
    "factual_true":      0.85,
    "factual_trap":      0.0,  # Excluida para este enfoque
    "out_of_domain":     0.0,  # Excluida siempre
}

def load_jsonl(path: str) -> list[dict]:
    """Lee un fichero JSONL y devuelve una lista de dicts, uno por línea."""
    questions = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # Ignorar líneas vacías
                questions.append(json.loads(line))
    return questions

def save_jsonl(conversations: list[dict], output_dir: str, filename: str) -> None:
    """Guarda la lista de conversaciones en un fichero JSONL, una conversación por línea."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, filename)
    with open(output_path, "w", encoding="utf-8") as f:
        for conv in conversations:
            f.write(json.dumps(conv, ensure_ascii=False) + "\n")
    print(f"{len(conversations)} conversaciones guardadas en {output_path}")

async def main():

  config = load_config("/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/GeneraciónBateríaPreguntas/cnf-Multiturn.yaml")

  #Parámetros de rutas y localización de ficheros
  input_cfg = config.get("input")
  input_path = input_cfg.get("single-turn_questions_path")
  docs_path = input_cfg.get("docs_path")


  output_path = config.get("output").get("path")

  questions = load_jsonl(input_path)

  api = config.get('api')
  token_api = api.get('key')
  model = api.get("model")

  client = OpenAI(api_key=token_api)
  client_async = AsyncOpenAI(api_key=token_api)

  prompts = config.get("prompts")

  doc_index = build_doc_index(docs_path=docs_path)


  seed_questions = select_seeds(all_questions = questions, client = client)
  print("20 MEJORES PREGUNTAS PARA USAR COMO SEMILLA")
  for i, question in enumerate(seed_questions, start = 1):
    print(f"Pregunta {i}: {question['metadata']['id']} \n—> score: {question['score']}")


  plot_seeds_ranking(seed_questions)
  top10_seeds = seed_questions[:10]

  conversations = await generate_all(top10_seeds, prompts, client_async, doc_index)
  print(f"\n {len(conversations)} conversaciones generadas")


  save_jsonl(conversations, output_dir=output_path, filename="golden_dataset_multi-turn.jsonl")

  jsonl_path = os.path.join(output_path, "golden_dataset_multi-turn.jsonl")
  export_conversations_readable(jsonl_path, output_dir=output_path)


In [ ]:
await main()